# Pandas Reward Calculation: Loops, Apply and Vectorization

Maintained companion for entry 8 of the efficient-Python notebook collection. Run all cells. Timings are measured on your machine, not a guaranteed 1000-fold gain. Original source code is preserved in a nonexecuting appendix. No API key or network is required after setup.

Install the tested packages in a fresh Python 3.13 environment:

```bash
python -m pip install numpy==2.3.3 pandas==3.1.0
```

In [ ]:
import sys, time, statistics
from importlib.metadata import version
import numpy as np
import pandas as pd
print("Python:", sys.version.split()[0])
print("NumPy:", version("numpy"), "pandas:", version("pandas"))

In [ ]:
def get_data(size=10000, seed=42):
    rng = np.random.default_rng(seed)
    return pd.DataFrame({
        "age": rng.integers(0, 100, size),
        "time_in_bed": rng.integers(0, 9, size),
        "pct_sleeping": rng.random(size),
        "favorite_food": rng.choice(["pizza", "ice-cream", "burger", "rice"], size),
        "hate_food": rng.choice(["milk", "vegetables", "eggs"], size),
    })
df = get_data()
assert len(df) == 10000 and len(get_data(7)) == 7
print(df.head().to_string(index=False))

In [ ]:
def reward_cal(row):
    if row["age"] >= 90:
        return row["favorite_food"]
    if row["time_in_bed"] > 5 and row["pct_sleeping"] > 0.5:
        return row["favorite_food"]
    return row["hate_food"]

def loop_method(frame):
    result = frame.copy()
    result["reward"] = pd.Series(index=result.index, dtype="str")
    for index, row in result.iterrows():
        result.loc[index, "reward"] = reward_cal(row)
    return result["reward"]

def apply_method(frame):
    if frame.empty:
        return frame["hate_food"].copy().rename("reward")
    return frame.apply(reward_cal, axis=1).rename("reward")

def vectorized_method(frame):
    result = frame["hate_food"].copy().rename("reward")
    mask = ((frame["time_in_bed"] > 5) & (frame["pct_sleeping"] > 0.5)) | (frame["age"] >= 90)
    result.loc[mask] = frame.loc[mask, "favorite_food"]
    return result
print("All three methods use age >= 90, time_in_bed > 5 and pct_sleeping > 0.5.")

In [ ]:
methods = {"Loop": loop_method, "Apply": apply_method, "Vectorized": vectorized_method}
results, measured = {}, []
for name, method in methods.items():
    durations = []
    for repeat in range(3):
        start = time.perf_counter()
        results[name] = method(df)
        durations.append(time.perf_counter() - start)
    measured.append({"method": name, "median_seconds": statistics.median(durations)})
timings = pd.DataFrame(measured)
print(timings.to_string(index=False))
print("Measured loop/vectorized ratio:", round(measured[0]["median_seconds"] / measured[2]["median_seconds"], 2))

In [ ]:
from itertools import product
from pandas.testing import assert_series_equal, assert_frame_equal
expected = [favorite if age >= 90 or (hours > 5 and fraction > 0.5) else disliked
            for age, hours, fraction, favorite, disliked in df.itertuples(index=False, name=None)]
for result in results.values():
    assert result.tolist() == expected
for result in list(results.values())[1:]:
    assert_series_equal(result, results["Loop"])
edges = pd.DataFrame(list(product([89, 90, 91], [5, 6], [0.49, 0.5, 0.51])),
                     columns=["age", "time_in_bed", "pct_sleeping"])
edges["favorite_food"] = "favorite"
edges["hate_food"] = "disliked"
baseline = apply_method(edges)
assert_series_equal(loop_method(edges), baseline)
assert_series_equal(vectorized_method(edges), baseline)
assert_series_equal(vectorized_method(edges.iloc[:0]), apply_method(edges.iloc[:0]))
assert_frame_equal(df, get_data())
assert df["pct_sleeping"].between(0, 1, inclusive="left").all()
print("PASS: all 10000 rows, all 18 boundary combinations, empty frame, deterministic fixture and unchanged inputs.")

## Maintenance and original-code appendix

The original generator ignored its size argument, drew one integer for every sleeping fraction and one disliked food for every row. The maintained generator uses independent fractions and choices per row, honors size and fixes the vectorized age boundary to match the row function. These synthetic rows are teaching data. Every original block follows verbatim; historical hard-coded timings are not rerun as if measured.

```python
def get_data(size= 10000):
    df = pd.DataFrame()
    size = 10000
    df['age'] = np.random.randint(0,100,size)
    df['time_in_bed'] = np.random.randint(0,9,size)
    df['pct_sleeping'] = np.random.randint(size)
    df['favorite_food'] = np.random.choice(['pizza','ice-cream','burger','rice'], size)
    df['hate_food'] = np.random.choice(['milk','vegetables','eggs'])
    return df
```

```python
df = get_data()
df.head()
```

```python
def reward_cal(row):
    if row['age'] >=90:
        return row['favorite_food'] 
    if (row['time_in_bed'] > 5) & (row['pct_sleeping']>0.5):
        return row['favorite_food']
    return row['hate_food']
```

```python
%%timeit

for index, row in df.iterrows():
   df.loc[index,'reward'] = reward_cal(row)
```

```python
%%timeit
df['reward'] = df.apply(reward_cal, axis = 1)
```

```python
%%timeit

df['reward'] = df['hate_food']
df.loc[((df['pct_sleeping']>0.5) &(df['time_in_bed']>5))| (df['age']>90), 'reward'] = df['favorite_food']
```

```python
results = pd.DataFrame(
    [
        ["Loop", 15800],
        ["apply", 535],
        ['vectorized', 5.8]
        
    ],
    columns = ['method', 'run_time']

)

results.set_index('method')['run_time'].plot(kind='bar')
```